# Reasoning Techniques (Deep Research)

The Deep Research pattern implements the **generate → research → reflect → finalize** loop: an agent generates search queries, gathers information, reflects on knowledge gaps, and iteratively refines until it can synthesize a comprehensive answer.

This is the architecture behind tools like Perplexity, Google Deep Research, and the open-source `gemini-fullstack-langgraph-quickstart`.

## Implementation with Flyte v2

This notebook reimplements the LangGraph `StateGraph` DeepSearch pattern from Chapter 17 using **Flyte v2 primitives only**.

#### LangGraph vs Flyte v2 — Key Differences

| Aspect | LangGraph | Flyte v2 |
|--------|-----------|----------|
| **Graph definition** | `StateGraph` with `add_node` / `add_conditional_edges` | Plain Python task composition with conditionals |
| **State type** | `OverallState` TypedDict | Typed `ResearchState` dataclass — serializable, inspectable |
| **Conditional routing** | `add_conditional_edges("reflection", evaluate_research, ...)` | Regular Python `if` inside the orchestrator task |
| **Checkpointing** | LangGraph checkpointer | `@flyte.trace` per LLM call — resumes from last checkpoint |
| **Live progress** | LangSmith traces | `flyte.report` HTML tab updated each iteration |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic duckduckgo-search

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta

import anthropic
import flyte
import flyte.report

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="reasoning-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.40.0", "duckduckgo-search>=6.0.0")
)

reasoning_env = flyte.TaskEnvironment(
    name="reasoning_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
    reusable=flyte.ReusePolicy(
        replicas=(1, 4),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=10),
    ),
)

### 4. Define data models

The LangGraph `StateGraph` used an `OverallState` TypedDict to pass state between nodes. In Flyte v2, `ResearchState` is a typed dataclass that is:
- Automatically serialized between task retries
- Visible as structured output in the UI (see each field's value without log parsing)
- Passable across task boundaries with full type safety

In [ ]:
@dataclass
class SearchResult:
    title: str
    snippet: str
    url: str = ""


@dataclass
class ResearchState:
    """Replaces LangGraph OverallState TypedDict."""
    question: str
    queries: list[str] = field(default_factory=list)
    search_results: list[SearchResult] = field(default_factory=list)
    knowledge_gaps: list[str] = field(default_factory=list)
    iteration: int = 0
    sufficient: bool = False

    def results_summary(self) -> str:
        if not self.search_results:
            return "No results gathered yet."
        return "\n\n".join(
            f"[{r.title}]\n{r.snippet}" for r in self.search_results
        )


@dataclass
class ResearchResult:
    """Final output of the Deep Research pipeline."""
    question: str
    answer: str
    iterations: int
    total_queries: int
    total_results: int

### 5. Define traced LLM + search helpers

Each helper is decorated with `@flyte.trace` — equivalent to individual LangGraph nodes, but as checkpoints within a single task. On pod failure mid-research, execution resumes from the last successful trace rather than starting over from query generation.

The LangGraph example used conditional edges: `add_conditional_edges("reflection", evaluate_research, ["web_research", "finalize_answer"])`. In Flyte v2, this becomes a plain `if state.sufficient` check inside the orchestrator loop — same semantics, no graph boilerplate.

In [ ]:
@flyte.trace
async def _generate_queries(question: str, previous_gaps: list[str]) -> list[str]:
    """Generate search queries. Replaces LangGraph generate_query node."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    gaps_context = ""
    if previous_gaps:
        gaps_context = "\nKnowledge gaps to address:\n" + "\n".join(f"- {g}" for g in previous_gaps)
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        system=(
            "Generate 2-3 focused web search queries for the given research question. "
            "Return ONLY the queries, one per line, no numbering or bullets."
        ),
        messages=[{"role": "user", "content": f"Research question: {question}{gaps_context}"}],
    )
    queries = [q.strip() for q in response.content[0].text.strip().splitlines() if q.strip()]
    return queries[:3]


@flyte.trace
async def _web_research(queries: list[str]) -> list[SearchResult]:
    """Execute searches. Replaces LangGraph web_research node."""
    import asyncio
    from duckduckgo_search import DDGS

    results: list[SearchResult] = []
    ddgs = DDGS()
    for query in queries:
        try:
            hits = ddgs.text(query, max_results=2)
            for hit in (hits or []):
                results.append(SearchResult(
                    title=hit.get("title", ""),
                    snippet=hit.get("body", ""),
                    url=hit.get("href", ""),
                ))
        except Exception:
            pass
        await asyncio.sleep(0.5)  # rate limit
    return results


@flyte.trace
async def _reflection(question: str, state: ResearchState) -> tuple[bool, list[str]]:
    """
    Evaluate gathered knowledge and identify gaps.
    Replaces LangGraph reflection + evaluate_research nodes.
    Returns (sufficient, knowledge_gaps).
    """
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system=(
            "Evaluate whether the gathered research is sufficient to answer the question comprehensively.\n"
            "Respond with exactly TWO sections:\n"
            "SUFFICIENT: yes or no\n"
            "GAPS: list up to 3 missing topics (one per line), or NONE if sufficient."
        ),
        messages=[{
            "role": "user",
            "content": (
                f"Research question: {question}\n\n"
                f"Gathered information:\n{state.results_summary()}"
            ),
        }],
    )
    text = response.content[0].text.strip()
    lines = text.splitlines()
    sufficient = False
    gaps: list[str] = []
    in_gaps = False
    for line in lines:
        if line.upper().startswith("SUFFICIENT:"):
            sufficient = "yes" in line.lower()
        elif line.upper().startswith("GAPS:"):
            in_gaps = True
            remainder = line.split(":", 1)[1].strip()
            if remainder and remainder.upper() != "NONE":
                gaps.append(remainder)
        elif in_gaps and line.strip() and line.strip().upper() != "NONE":
            gaps.append(line.strip())
    return sufficient, gaps


@flyte.trace
async def _finalize_answer(question: str, state: ResearchState) -> str:
    """Synthesize final answer from all gathered research. Replaces LangGraph finalize_answer node."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=(
            "You are a research synthesizer. Using the gathered information, write a comprehensive, "
            "well-structured answer to the research question. Cite specific facts from the research. "
            "Be accurate and concise."
        ),
        messages=[{
            "role": "user",
            "content": (
                f"Research question: {question}\n\n"
                f"Gathered information:\n{state.results_summary()}"
            ),
        }],
    )
    return response.content[0].text.strip()

### 6. Define the Deep Research orchestrator task

The LangGraph version compiled a `StateGraph` and called `app.stream(inputs)`. In Flyte v2, the orchestration loop is plain Python — the same `if state.sufficient` conditional that LangGraph expressed as `add_conditional_edges`. No graph definition, no compilation step, same control flow.

In [ ]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


@reasoning_env.task(
    retries=2,
    timeout=timedelta(minutes=20),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def deep_research(
    question: str,
    max_iterations: int = 3,
) -> ResearchResult:
    """
    Deep Research loop: generate queries → search → reflect → refine → finalize.

    Replaces the LangGraph StateGraph from gemini-fullstack-langgraph-quickstart:
      generate_query → web_research → reflection →
        [evaluate_research: loop back or finalize_answer]

    The conditional routing in LangGraph (add_conditional_edges) becomes
    plain Python: `if state.sufficient: break`.
    """
    state = ResearchState(question=question)
    report_sections: list[str] = []

    for i in range(max_iterations):
        state.iteration = i + 1
        label = f"Iteration {i + 1} / {max_iterations}"

        # generate_query node
        queries = await _generate_queries(
            question=question,
            previous_gaps=state.knowledge_gaps,
        )
        state.queries.extend(queries)

        # web_research node
        new_results = await _web_research(queries=queries)
        state.search_results.extend(new_results)

        # reflection + evaluate_research nodes
        sufficient, gaps = await _reflection(question=question, state=state)
        state.sufficient = sufficient
        state.knowledge_gaps = gaps

        color = "green" if sufficient else "orange"
        status = "SUFFICIENT" if sufficient else "NEEDS MORE RESEARCH"
        report_sections.append(
            f"<section><h2>{label} — <span style='color:{color}'>{status}</span></h2>"
            f"<h3>Queries ({len(queries)})</h3><ul>"
            + "".join(f"<li>{_html_escape(q)}</li>" for q in queries)
            + f"</ul><h3>Results ({len(new_results)} new)</h3><ul>"
            + "".join(f"<li><b>{_html_escape(r.title)}</b>: {_html_escape(r.snippet[:120])}...</li>" for r in new_results)
            + (f"<h3>Knowledge Gaps</h3><ul>" + "".join(f"<li>{_html_escape(g)}</li>" for g in gaps) + "</ul>" if gaps else "")
            + "</section><hr/>"
        )

        report_html = (
            "<!DOCTYPE html><html><head><style>"
            "body{font-family:-apple-system,monospace;padding:1.5em;max-width:900px;margin:auto}"
            "h1{border-bottom:2px solid #333;padding-bottom:.4em}"
            "</style></head><body>"
            f"<h1>Deep Research — {_html_escape(question[:60])}...</h1>"
            f"<p>Iteration: <strong>{i+1}</strong> | Status: <strong>{status}</strong></p>"
            + "\n".join(report_sections)
            + "</body></html>"
        )
        await flyte.report.replace.aio(report_html)
        await flyte.report.flush.aio()

        # Conditional edge: if sufficient → finalize, else loop
        if state.sufficient:
            break

    # finalize_answer node
    answer = await _finalize_answer(question=question, state=state)

    return ResearchResult(
        question=question,
        answer=answer,
        iterations=state.iteration,
        total_queries=len(state.queries),
        total_results=len(state.search_results),
    )

### 7. Run locally

In [ ]:
run = flyte.run(
    deep_research,
    question="What are the main differences between Flyte and Apache Airflow for ML pipelines?",
    max_iterations=3,
)
run.wait()
result: ResearchResult = run.outputs()[0]

print(f"Question: {result.question}")
print(f"Iterations: {result.iterations}, Queries: {result.total_queries}, Results: {result.total_results}")
print("\n" + "=" * 60)
print(result.answer)

### Running remotely

Remote execution adds the live `report` tab — watch each iteration's queries, results, and knowledge gaps update in real time as the agent researches.

In [ ]:
run = flyte.run(
    deep_research,
    question="What are the main differences between Flyte and Apache Airflow for ML pipelines?",
    max_iterations=3,
)
run.wait()
result = run.outputs()[0]
print(f"Iterations: {result.iterations}")
print(result.answer)